# Sentiment Analysis with Naive Bayes

This notebook builds a sentiment classifier for Google Play Store reviews using Naive Bayes models.

## Goals
1. Load and inspect the dataset.
2. Clean and vectorize text with Bag of Words (`CountVectorizer`).
3. Train and compare `GaussianNB`, `MultinomialNB`, and `BernoulliNB`.
4. Try to improve results with `RandomForestClassifier`.
5. Save the best model.
6. Explore another alternative model (`LogisticRegression`).

In [12]:
# Install dependencies in the active notebook kernel (run this once)
%pip install -q pandas numpy scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import pickle
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [14]:
# Step 1: Load dataset
url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"
local_path = "playstore_reviews.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    df = pd.read_csv(url)

df.head(), df.shape

(          package_name                                             review  \
 0  com.facebook.katana   privacy at least put some option appear offli...   
 1  com.facebook.katana   messenger issues ever since the last update, ...   
 2  com.facebook.katana   profile any time my wife or anybody has more ...   
 3  com.facebook.katana   the new features suck for those of us who don...   
 4  com.facebook.katana   forced reload on uploading pic on replying co...   
 
    polarity  
 0         0  
 1         0  
 2         0  
 3         0  
 4         0  ,
 (891, 3))

In [15]:
# Step 2: Text processing
# - strip and lowercase text
# - remove package_name
# - split train/test
# - vectorize with CountVectorizer

df["review"] = df["review"].astype(str).str.strip().str.lower()
df = df.drop("package_name", axis=1)

X = df["review"]
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = CountVectorizer(stop_words="english")

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

X_train_vec.shape, X_test_vec.shape

((712, 3310), (179, 3310))

In [16]:
# Step 3: Train and compare Naive Bayes implementations
models = {
    "GaussianNB": GaussianNB(),
    "MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB(),
}

results = []
trained_models = {}

for name, model in models.items():
    if name == "GaussianNB":
        # GaussianNB expects dense arrays
        model.fit(X_train_vec.toarray(), y_train)
        preds = model.predict(X_test_vec.toarray())
    else:
        # Multinomial and Bernoulli work naturally with count/sparse text features
        model.fit(X_train_vec, y_train)
        preds = model.predict(X_test_vec)

    acc = accuracy_score(y_test, preds)
    results.append({"model": name, "accuracy": acc})
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(by="accuracy", ascending=False)
results_df

,model,accuracy
1,MultinomialNB,0.815642
0,GaussianNB,0.804469
2,BernoulliNB,0.770950


In [17]:
# Evaluate the best Naive Bayes model in detail
best_nb_name = results_df.iloc[0]["model"]
best_nb_model = trained_models[best_nb_name]

if best_nb_name == "GaussianNB":
    best_nb_preds = best_nb_model.predict(X_test_vec.toarray())
else:
    best_nb_preds = best_nb_model.predict(X_test_vec)

print(f"Best Naive Bayes model: {best_nb_name}")
print("Accuracy:", accuracy_score(y_test, best_nb_preds))
print("\nClassification report:\n")
print(classification_report(y_test, best_nb_preds))
print("Confusion matrix:\n", confusion_matrix(y_test, best_nb_preds))

Best Naive Bayes model: MultinomialNB
Accuracy: 0.8156424581005587

Classification report:

              precision    recall  f1-score   support

           0       0.84      0.90      0.87       126
           1       0.73      0.60      0.66        53

    accuracy                           0.82       179
   macro avg       0.79      0.75      0.77       179
weighted avg       0.81      0.82      0.81       179

Confusion matrix:
 [[114  12]
 [ 21  32]]


In [18]:
# Step 4: Try optimization with Random Forest
# RandomForest needs dense numeric matrix in this setup.
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train_vec.toarray(), y_train)
rf_preds = rf.predict(X_test_vec.toarray())
rf_acc = accuracy_score(y_test, rf_preds)

print("Random Forest accuracy:", rf_acc)
print("\nClassification report:\n")
print(classification_report(y_test, rf_preds))

Random Forest accuracy: 0.8156424581005587

Classification report:

              precision    recall  f1-score   support

           0       0.89      0.84      0.87       126
           1       0.67      0.75      0.71        53

    accuracy                           0.82       179
   macro avg       0.78      0.80      0.79       179
weighted avg       0.82      0.82      0.82       179



In [19]:
# Step 5: Save best model and vectorizer
os.makedirs("models", exist_ok=True)

best_overall_name = best_nb_name
best_overall_model = best_nb_model
best_overall_acc = accuracy_score(y_test, best_nb_preds)

if rf_acc > best_overall_acc:
    best_overall_name = "RandomForestClassifier"
    best_overall_model = rf
    best_overall_acc = rf_acc

with open("models/count_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("models/best_model.pkl", "wb") as f:
    pickle.dump(best_overall_model, f)

print("Saved vectorizer to models/count_vectorizer.pkl")
print(f"Saved best model ({best_overall_name}) to models/best_model.pkl")
print(f"Best overall accuracy: {best_overall_acc:.4f}")

Saved vectorizer to models/count_vectorizer.pkl
Saved best model (MultinomialNB) to models/best_model.pkl
Best overall accuracy: 0.8156


In [20]:
# Step 6: Explore another alternative model (Logistic Regression)
log_reg = LogisticRegression(max_iter=2000, random_state=42)
log_reg.fit(X_train_vec, y_train)
log_reg_preds = log_reg.predict(X_test_vec)
log_reg_acc = accuracy_score(y_test, log_reg_preds)

print("Logistic Regression accuracy:", log_reg_acc)
print("\nClassification report:\n")
print(classification_report(y_test, log_reg_preds))

Logistic Regression accuracy: 0.8324022346368715

Classification report:

              precision    recall  f1-score   support

           0       0.91      0.84      0.88       126
           1       0.68      0.81      0.74        53

    accuracy                           0.83       179
   macro avg       0.80      0.83      0.81       179
weighted avg       0.85      0.83      0.84       179



In [21]:
# Final model comparison
comparison_df = pd.DataFrame([
    {"model": "GaussianNB", "accuracy": float(results_df[results_df["model"] == "GaussianNB"]["accuracy"].values[0])},
    {"model": "MultinomialNB", "accuracy": float(results_df[results_df["model"] == "MultinomialNB"]["accuracy"].values[0])},
    {"model": "BernoulliNB", "accuracy": float(results_df[results_df["model"] == "BernoulliNB"]["accuracy"].values[0])},
    {"model": "RandomForestClassifier", "accuracy": rf_acc},
    {"model": "LogisticRegression", "accuracy": log_reg_acc},
]).sort_values(by="accuracy", ascending=False)

comparison_df

,model,accuracy
4,LogisticRegression,0.832402
3,RandomForestClassifier,0.815642
1,MultinomialNB,0.815642
0,GaussianNB,0.804469
2,BernoulliNB,0.770950
